## Step 1: Import SageMaker libraries and set session variables
### What this does

Sets up the SageMaker SDK, execution role, session, region, and output paths.

In [1]:
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.transformer import Transformer

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name

bucket = "retailpulse-team3-ads508"

train_key = "processed-data/train/train.csv"
val_key   = "processed-data/validation/validation.csv"
test_key  = "processed-data/test/test.csv"

xgb_train_key = "training-data/xgboost/train/train_xgb.csv"
xgb_val_key   = "training-data/xgboost/validation/validation_xgb.csv"
xgb_test_key  = "training-data/xgboost/test/test_xgb.csv"

output_path = f"s3://{bucket}/model-output/xgboost/"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Step 2: Load the prepared split files
### What this does

Reads the train, validation, and test data that you created in the previous module.

In [2]:
import boto3
import pandas as pd
import io

s3 = boto3.client("s3")

def read_csv_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj["Body"].read()))

train_df = read_csv_from_s3(bucket, train_key)
val_df   = read_csv_from_s3(bucket, val_key)
test_df  = read_csv_from_s3(bucket, test_key)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2152, 111)
Validation shape: (461, 111)
Test shape: (462, 111)


## Step 3: Reformat for built-in XGBoost
### What this does

SageMaker built-in XGBoost expects:

- CSV format
- no header
- target column first

AWS documents this explicitly for CSV training data.

In [3]:
target_col = "weekly_quantity"

def make_xgb_ready(df, target):
    cols = [target] + [c for c in df.columns if c != target]
    return df[cols].copy()

train_xgb = make_xgb_ready(train_df, target_col)
val_xgb   = make_xgb_ready(val_df, target_col)
test_xgb  = make_xgb_ready(test_df, target_col)

print(train_xgb.head())

   weekly_quantity  PurchaseYear  PurchaseWeek  weekly_revenue  avg_price  \
0               37          2010            23           98.80   3.004286   
1               15          2010            23           53.13   3.795714   
2               56          2010            23           36.64   1.097500   
3               48          2010            23            5.76   0.120000   
4               87          2010            23          199.98   2.220000   

   invoice_count  purchase_month  quarter week_start_date  lag_1_week_qty  \
0              7               6        2      2010-06-07            14.0   
1              7               6        2      2010-06-07            72.0   
2              4               6        2      2010-06-07            10.0   
3              2               6        2      2010-06-07            24.0   
4              9               6        2      2010-06-07            64.0   

   ...  StockCode_84945  StockCode_84946  StockCode_84947  StockCode_84978

## Step 4: Remove columns not needed for built-in training
### What this does

If you still have date columns like week_start_date, remove them unless they were already encoded numerically.

In [4]:
drop_if_present = ["week_start_date"]

for col in drop_if_present:
    if col in train_xgb.columns:
        train_xgb = train_xgb.drop(columns=[col])
    if col in val_xgb.columns:
        val_xgb = val_xgb.drop(columns=[col])
    if col in test_xgb.columns:
        test_xgb = test_xgb.drop(columns=[col])

print(train_xgb.columns.tolist()[:10])
print("Final training columns:", len(train_xgb.columns))

['weekly_quantity', 'PurchaseYear', 'PurchaseWeek', 'weekly_revenue', 'avg_price', 'invoice_count', 'purchase_month', 'quarter', 'lag_1_week_qty', 'lag_2_week_qty']
Final training columns: 110


## Step 5: Save headerless CSV files locally
### What this does

Built-in XGBoost requires CSV files in the expected structure, so save without headers and without index.

In [6]:
local_train_xgb = "/home/sagemaker-user/train_xgb.csv"
local_val_xgb   = "/home/sagemaker-user/validation_xgb.csv"
local_test_xgb  = "/home/sagemaker-user/test_xgb.csv"

train_xgb.to_csv(local_train_xgb, index=False, header=False)
val_xgb.to_csv(local_val_xgb, index=False, header=False)
test_xgb.to_csv(local_test_xgb, index=False, header=False)

print("Saved local XGBoost-formatted files.")

Saved local XGBoost-formatted files.


## Step 6: Upload the XGBoost-formatted files to S3
### What this does

Stores the final training-ready files in S3.

In [7]:
s3.upload_file(local_train_xgb, bucket, xgb_train_key)
s3.upload_file(local_val_xgb, bucket, xgb_val_key)
s3.upload_file(local_test_xgb, bucket, xgb_test_key)

train_s3_uri = f"s3://{bucket}/{xgb_train_key}"
val_s3_uri   = f"s3://{bucket}/{xgb_val_key}"
test_s3_uri  = f"s3://{bucket}/{xgb_test_key}"

print(train_s3_uri)
print(val_s3_uri)
print(test_s3_uri)

s3://retailpulse-team3-ads508/training-data/xgboost/train/train_xgb.csv
s3://retailpulse-team3-ads508/training-data/xgboost/validation/validation_xgb.csv
s3://retailpulse-team3-ads508/training-data/xgboost/test/test_xgb.csv


## Step 7: Retrieve the SageMaker XGBoost container
### What this does

Gets the correct built-in XGBoost image URI for your region.

In [8]:
container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

print(container)

683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


## Step 8: Define the XGBoost estimator
### What this does

Creates the training job configuration.

### Why these settings

For your project, start simple:

- instance_count=1
- instance_type="ml.m5.large"

This is reasonable for a student project and a structured tabular dataset. SageMaker built-in XGBoost runs directly from this estimator setup.

In [9]:
xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size=20,
    output_path=output_path,
    sagemaker_session=session
)

## Step 9: Set hyperparameters
### What this does

Passes the algorithm parameters to XGBoost.

### Recommended first-pass parameters

These are good starting values for a regression problem:

- objective="reg:squarederror"
- num_round=200
- max_depth=6
- eta=0.1
- subsample=0.8
- colsample_bytree=0.8
- min_child_weight=3

AWS documents the supported XGBoost hyperparameters and regression objectives.

In [10]:
xgb_estimator.set_hyperparameters(
    objective="reg:squarederror",
    num_round=200,
    max_depth=6,
    eta=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric="rmse"
)

## Step 10: Define training inputs
### What this does

Specifies the S3 training and validation channels.

In [11]:
train_input = TrainingInput(
    s3_data=train_s3_uri,
    content_type="text/csv"
)

validation_input = TrainingInput(
    s3_data=val_s3_uri,
    content_type="text/csv"
)

## Step 11: Launch the SageMaker training job
### What this does

Starts the actual managed training job in SageMaker Studio.

In [12]:
xgb_estimator.fit(
    inputs={
        "train": train_input,
        "validation": validation_input
    },
    wait=True
)

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-25-20-35-22-584


2026-03-25 20:35:25 Starting - Starting the training job...
2026-03-25 20:35:42 Starting - Preparing the instances for training...
2026-03-25 20:36:07 Downloading - Downloading input data...
2026-03-25 20:36:57 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-25 20:38:08.423 ip-10-0-231-17.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-25 20:38:08.509 ip-10-0-231-17.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-25:20:38:08:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-25:20:38:08:INFO] Failed to parse hyperparameter eval_metric value rmse to J

## Step 12: Set up batch transform for test prediction
### What this does

Uses the trained model to generate predictions on the test set.

### Important

For inference, the built-in algorithm should get features only, not the target. So remove the first column from the test file before prediction.

In [13]:
# Remove target column for test inference input
X_test_only = test_xgb.drop(columns=[target_col])

local_test_features = "/home/sagemaker-user/test_features_only.csv"
X_test_only.to_csv(local_test_features, index=False, header=False)

test_features_key = "training-data/xgboost/test/test_features_only.csv"
s3.upload_file(local_test_features, bucket, test_features_key)

test_features_s3_uri = f"s3://{bucket}/{test_features_key}"
print(test_features_s3_uri)

s3://retailpulse-team3-ads508/training-data/xgboost/test/test_features_only.csv


## Step 13: Run batch transform
### What this does

Creates prediction output files in S3.

In [16]:
# Ensure all columns are numeric for SageMaker built-in XGBoost
for df_name, df_ in [("train_xgb", train_xgb), ("val_xgb", val_xgb), ("test_xgb", test_xgb)]:
    bool_cols = df_.select_dtypes(include=["bool"]).columns
    if len(bool_cols) > 0:
        print(f"{df_name} bool columns:", list(bool_cols))
        df_[bool_cols] = df_[bool_cols].astype(int)

    object_cols = df_.select_dtypes(include=["object"]).columns
    if len(object_cols) > 0:
        print(f"{df_name} object columns before conversion:", list(object_cols))
        for col in object_cols:
            df_[col] = pd.to_numeric(df_[col], errors="raise")

# Recreate features-only test file
X_test_only = test_xgb.drop(columns=[target_col]).copy()

bool_cols = X_test_only.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_test_only[bool_cols] = X_test_only[bool_cols].astype(int)

object_cols = X_test_only.select_dtypes(include=["object"]).columns
if len(object_cols) > 0:
    for col in object_cols:
        X_test_only[col] = pd.to_numeric(X_test_only[col], errors="raise")

print("Final dtypes in X_test_only:")
print(X_test_only.dtypes.value_counts())

# Save again
local_train_xgb = "/home/sagemaker-user/train_xgb.csv"
local_val_xgb   = "/home/sagemaker-user/validation_xgb.csv"
local_test_xgb  = "/home/sagemaker-user/test_xgb.csv"
local_test_features = "/home/sagemaker-user/test_features_only.csv"

train_xgb.to_csv(local_train_xgb, index=False, header=False)
val_xgb.to_csv(local_val_xgb, index=False, header=False)
test_xgb.to_csv(local_test_xgb, index=False, header=False)
X_test_only.to_csv(local_test_features, index=False, header=False)

# Upload again
s3.upload_file(local_train_xgb, bucket, xgb_train_key)
s3.upload_file(local_val_xgb, bucket, xgb_val_key)
s3.upload_file(local_test_xgb, bucket, xgb_test_key)

test_features_key = "training-data/xgboost/test/test_features_only.csv"
s3.upload_file(local_test_features, bucket, test_features_key)

test_features_s3_uri = f"s3://{bucket}/{test_features_key}"
print("Uploaded test features file:", test_features_s3_uri)

train_xgb bool columns: ['StockCode_15036', 'StockCode_16014', 'StockCode_17003', 'StockCode_20668', 'StockCode_20712', 'StockCode_20713', 'StockCode_20719', 'StockCode_20723', 'StockCode_20724', 'StockCode_20725', 'StockCode_20727', 'StockCode_20728', 'StockCode_20971', 'StockCode_21080', 'StockCode_21108', 'StockCode_21137', 'StockCode_21166', 'StockCode_21175', 'StockCode_21181', 'StockCode_21212', 'StockCode_21213', 'StockCode_21232', 'StockCode_21326', 'StockCode_21498', 'StockCode_21703', 'StockCode_21731', 'StockCode_21733', 'StockCode_21790', 'StockCode_21791', 'StockCode_21891', 'StockCode_21915', 'StockCode_21929', 'StockCode_21931', 'StockCode_21975', 'StockCode_21977', 'StockCode_21982', 'StockCode_22086', 'StockCode_22151', 'StockCode_22178', 'StockCode_22189', 'StockCode_22197', 'StockCode_22355', 'StockCode_22382', 'StockCode_22383', 'StockCode_22384', 'StockCode_22386', 'StockCode_22411', 'StockCode_22423', 'StockCode_22457', 'StockCode_22469', 'StockCode_22470', 'Stock

In [17]:
transformer = xgb_estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/batch-output/xgboost/"
)

transformer.transform(
    data=test_features_s3_uri,
    content_type="text/csv",
    split_type="Line"
)

transformer.wait()

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-03-25-20-49-41-899
INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-03-25-20-49-42-880


................................/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-25:20:54:58:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-25:20:54:58:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-25:20:54:58:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package 

## Step 14: Read predictions from S3
### What this does

Loads the batch prediction results so you can evaluate them.

In [18]:
batch_output_key = "batch-output/xgboost/test_features_only.csv.out"

pred_obj = s3.get_object(Bucket=bucket, Key=batch_output_key)
predictions = pd.read_csv(io.BytesIO(pred_obj["Body"].read()), header=None)

y_pred = predictions[0].values
y_true = test_xgb[target_col].values

print("Predictions shape:", predictions.shape)
print("True values shape:", y_true.shape)

Predictions shape: (462, 1)
True values shape: (462,)


## Step 15: Evaluate the model
### What this does

Computes the evaluation metrics you will discuss in the design document.

In [19]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("SageMaker XGBoost Test Metrics")
print("RMSE:", round(rmse, 4))
print("MAE :", round(mae, 4))
print("R^2 :", round(r2, 4))

SageMaker XGBoost Test Metrics
RMSE: 154.7636
MAE : 50.1849
R^2 : 0.8166


## Step 16: Save the results table
### What this does

Creates a clean output table for the design document and GitHub.

In [20]:
results_df = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "R2"],
    "Value": [rmse, mae, r2]
})

results_df

,Metric,Value
0,RMSE,154.763562
1,MAE,50.184867
2,R2,0.816616


## Step 17: Save results locally and optionally upload to S3

In [21]:
results_file = "/home/sagemaker-user/xgboost_test_results.csv"
results_df.to_csv(results_file, index=False)

results_key = "model-output/xgboost/xgboost_test_results.csv"
s3.upload_file(results_file, bucket, results_key)

print(f"s3://{bucket}/{results_key}")

s3://retailpulse-team3-ads508/model-output/xgboost/xgboost_test_results.csv
